## Task 1

In [4]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

model_ids = {
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "SmolLM3-3B": "HuggingFaceTB/SmolLM3-3B"
}

test_examples = [
    {
        "task": "Logical Negation",
        "prompt": "Which of the following is NOT a botanical berry: banana, grape, or peach? Answer with one word.",
        "expected": "{peach}"
    },
    {
        "task": "Ambiguous Reference",
        "prompt": "The trophy didn't fit into the brown suitcase because it was too large. What was too large? Answer with one noun phrase only.",
        "expected": "{the trophy, trophy}"
    },
    {
        "task": "Constrained Output",
        "prompt": "Write these three words in alphabetical order, separated by commas, with no spaces: green, blue, red.",
        "expected": "{blue,green,red}"
    },
    {
        "task": "Data Extraction",
        "prompt": "ROI 2021: 15%. ROI 2022: 12%. Give the 2022 ROI as a decimal. Answer with numbers only.",
        "expected": "{0.12}"
    },
    {
        "task": "Counterfactual Reasoning",
        "prompt": "If all birds suddenly lost their wings, could they still fly naturally? Answer yes or no only.",
        "expected": "{No}"
    }
]

def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )

    model.eval()
    return tokenizer, model


def unload_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_response(model, tokenizer, prompt, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,   # key change
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False
        )

    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def run_evaluation_sequential(model_ids, test_examples):
    results_list = []

    for ex in test_examples:
        results_list.append({
            "Task": ex["task"],
            "Prompt": ex["prompt"],
            "Expected": ex["expected"]
        })

    for model_label, model_name in model_ids.items():
        print(f"Loading {model_label}...")
        tokenizer, model = load_model_and_tokenizer(model_name)

        for i, ex in enumerate(test_examples):
            try:
                response = generate_response(model, tokenizer, ex["prompt"])
            except torch.cuda.OutOfMemoryError:
                print(f"OOM on {model_label}, task={ex['task']}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                response = "[OOM]"

            results_list[i][model_label] = response

        unload_model(model, tokenizer)

    df = pd.DataFrame(results_list)
    return df


task1 = run_evaluation_sequential(model_ids, test_examples)
task1

Loading Qwen2.5-3B...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading SmolLM3-3B...


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

,Task,Prompt,Expected,Qwen2.5-3B,SmolLM3-3B
0,Logical Negation,Which of the following is NOT a botanical berr...,{peach},banana,peach
1,Ambiguous Reference,The trophy didn't fit into the brown suitcase ...,"{the trophy, trophy}",trophy,The trophy was too large.
2,Constrained Output,"Write these three words in alphabetical order,...","{blue,green,red}","green,blue,red","blue, green, red"
3,Data Extraction,ROI 2021: 15%. ROI 2022: 12%. Give the 2022 RO...,{0.12},0.12,"To convert the ROI from percentage to decimal,..."
4,Counterfactual Reasoning,"If all birds suddenly lost their wings, could ...",{No},No,No.


In [5]:
task1.to_csv('task1_results.csv', index=False)

## Task 2

In [18]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd


best_model_name = "SmolLM3-3B"
best_model_id = "HuggingFaceTB/SmolLM3-3B"

test_examples = [
    {
        "task": "Logical Negation",
        "prompt": "Which of the following is NOT a botanical berry: banana, grape, or peach? Answer with one word only.",
        "expected": "{peach}"
    },
    {
        "task": "Ambiguous Reference",
        "prompt": "The trophy didn't fit into the brown suitcase because it was too large. What was too large? Answer with one noun phrase only.",
        "expected": "{the trophy, trophy}"
    },
    {
        "task": "Constrained Output",
        "prompt": "Write these three words in alphabetical order, separated by commas, with no spaces: blue, green, red. Output only the final string.",
        "expected": "{blue,green,red}"
    },
    {
        "task": "Data Extraction",
        "prompt": "ROI 2021: 15%. ROI 2022: 12%. Give the 2022 ROI as a decimal. Answer with numbers only.",
        "expected": "{0.12}"
    },
    {
        "task": "Counterfactual Reasoning",
        "prompt": "If all birds suddenly lost their wings, could they still fly naturally? Answer yes or no only.",
        "expected": "{No}"
    }
]

def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    return tokenizer, model

def generate_response(model, tokenizer, prompt, max_new_tokens=40):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False
        )

    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return response
def build_zero_shot_prompt(user_prompt):
    return f"""Answer the following task exactly as instructed.

Task:
{user_prompt}
"""

def build_plan_and_solve_prompt(user_prompt):
    return f"""Solve the following problem in two stages.

Stage 1: Make a brief plan for how to answer correctly.
Stage 2: Use that plan to produce the final answer.

Important:
- Follow every constraint in the question exactly.
- Keep the final answer concise.
- After your reasoning, end with a line that starts with: Final Answer:

Question:
{user_prompt}
Output only the final revised answer, with no explanation.
"""

def build_self_refine_prompt(user_prompt):
    return f"""Answer the following task exactly as instructed.

Task:
{user_prompt}

Step 1: Draft an initial answer internally.
Step 2: Check whether it follows every constraint in the prompt.
Step 3: Revise if needed.

Output only the final revised answer, with no explanation.
"""

def extract_final_answer(response: str):
    match = re.search(r"Final Answer:\s*(.*)", response, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

def normalize_text(s):
    return s.strip().lower()

def check_correct(task, prediction, expected):
    pred = normalize_text(prediction)

    if task == "Logical Negation":
        return pred == "peach"
    elif task == "Ambiguous Reference":
        return pred in ["the trophy", "trophy"]
    elif task == "Constrained Output":
        return pred == "blue,green,red"
    elif task == "Data Extraction":
        return pred in ["0.12", ".12"]
    elif task == "Counterfactual Reasoning":
        return pred == "no"

    return False

def run_task2(model, tokenizer, test_examples):
    rows = []

    for ex in test_examples:
        prompts = {
            "Zero-shot": build_zero_shot_prompt(ex["prompt"]),
            "Plan-and-Solve": build_plan_and_solve_prompt(ex["prompt"]),
            "Self-Refine": build_self_refine_prompt(ex["prompt"])
        }

        row = {
            "Task": ex["task"],
            "Original Prompt": ex["prompt"],
            "Expected": ex["expected"]
        }

        for method_name, full_prompt in prompts.items():
            response = generate_response(model, tokenizer, full_prompt)

            if method_name == "Plan-and-Solve":
                response = extract_final_answer(response)

            row[f"{method_name} Output"] = response


        rows.append(row)

    return pd.DataFrame(rows)




In [19]:
tokenizer, model = load_model_and_tokenizer(best_model_id)
task2_df = run_task2(model, tokenizer, test_examples)
display(task2_df)


,Task,Original Prompt,Expected,Zero-shot Output,Plan-and-Solve Output,Self-Refine Output
0,Logical Negation,Which of the following is NOT a botanical berr...,{peach},peach,"Stage 1: To answer correctly, I need to unders...",peach
1,Ambiguous Reference,The trophy didn't fit into the brown suitcase ...,"{the trophy, trophy}",The trophy was too large.,"Stage 1: To answer correctly, I need to identi...",trophy
2,Constrained Output,"Write these three words in alphabetical order,...","{blue,green,red}","blue,green,red","Stage 1: To answer correctly, I need to list t...","blue, green, red"
3,Data Extraction,ROI 2021: 15%. ROI 2022: 12%. Give the 2022 RO...,{0.12},"To convert the ROI from percentage to decimal,...","Stage 1: To answer correctly, I need to conver...",0.12
4,Counterfactual Reasoning,"If all birds suddenly lost their wings, could ...",{No},"No\n\nIf all birds suddenly lost their wings, ...","Stage 1: To answer this question correctly, I ...",No


In [20]:
task2_df.to_csv('task2_results.csv', index=False)

## Task 3

In [28]:
import json
import random
import re
import gc
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

TEAM_NAME = "ChengFu"
OUTPUT_JSON = f"csci5541-s26-hw5-{TEAM_NAME}-3c.json"
SEED = 42
NUM_EXAMPLES = 30

model_ids = {
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "SmolLM3-3B": "HuggingFaceTB/SmolLM3-3B"
}

prompting_methods = ["zero-shot", "plan-and-solve", "self-refine"]

GEN_CONFIG = {
    "temperature": 0.0,
    "top_p": 1.0,
    "max_new_tokens": 40,
    "num_generations": 1,
    "seed": SEED
}

random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset("tau/commonsense_qa")
selected_examples = dataset["validation"].select(range(NUM_EXAMPLES))


def build_zero_shot_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful assistant."},
        {"role": "user", "content": f"""Choose the best answer to the multiple-choice question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Output constraints:
- Answer with exactly one letter.
- Allowed outputs: A, B, C, D, or E.
- Do not explain.
- Output only the final answer.
"""}
    ]


def build_plan_and_solve_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful reasoning assistant."},
        {"role": "user", "content": f"""Solve the task in two stages.

Stage 1: Briefly compare the answer choices against the question.
Stage 2: Decide which option best answers the question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Output constraints:
- Exactly one letter
- Allowed outputs: A, B, C, D, or E
- No explanation
"""}
    ]


def build_self_refine_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful assistant that checks its own output."},
        {"role": "user", "content": f"""Choose the best answer to the multiple-choice question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Instructions:
1. Draft an answer internally.
2. Check whether the answer matches the question and choices.
3. Check whether the answer follows the output constraint.
4. Output only the final revised answer.

Output constraints:
- Exactly one letter
- Allowed outputs: A, B, C, D, or E
- No explanation
"""}
    ]


def build_messages(method, question, choices):
    if method == "zero-shot":
        return build_zero_shot_prompt(question, choices)
    if method == "plan-and-solve":
        return build_plan_and_solve_prompt(question, choices)
    if method == "self-refine":
        return build_self_refine_prompt(question, choices)
    raise ValueError(f"Unknown prompting method: {method}")


def load_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    return tokenizer, model


def generate_response(model, tokenizer, messages, max_new_tokens):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_time = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return response, elapsed_time


def strict_parse_choice(raw_output, method):
    text = raw_output.strip()

    if method == "plan-and-solve":
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        for line in reversed(lines):
            m = re.fullmatch(r"Final Answer:\s*([ABCDE])\s*", line, flags=re.IGNORECASE)
            if m:
                return m.group(1).upper()
        return None

    if text in {"A", "B", "C", "D", "E"}:
        return text
    return None


def compute_scores(parsed_pred, gold):
    format_valid = int(parsed_pred in {"A", "B", "C", "D", "E"})
    exact_match = int(parsed_pred == gold) if parsed_pred is not None else 0
    return format_valid, exact_match


def unload_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


all_results = []
example_id = 1

for model_label, model_path in model_ids.items():
    print(f"Loading model: {model_label}")
    tokenizer, model = load_model_and_tokenizer(model_path)

    for i, ex in enumerate(selected_examples):
        question = ex["question"]
        choice_map = dict(zip(ex["choices"]["label"], ex["choices"]["text"]))
        choices = {
            "A": choice_map["A"],
            "B": choice_map["B"],
            "C": choice_map["C"],
            "D": choice_map["D"],
            "E": choice_map["E"],
        }
        gold = ex["answerKey"]
        dataset_item_id = f"commonsenseqa_val_{i:03d}"

        for method in prompting_methods:
            messages = build_messages(method, question, choices)

            try:
                raw_output, response_time_sec = generate_response(
                    model=model,
                    tokenizer=tokenizer,
                    messages=messages,
                    max_new_tokens=GEN_CONFIG["max_new_tokens"]
                )
            except torch.cuda.OutOfMemoryError:
                print(f"OOM at model={model_label}, example={i}, method={method}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                raw_output = ""
                response_time_sec = None

            parsed_pred = strict_parse_choice(raw_output, method)
            format_valid, exact = compute_scores(parsed_pred, gold)

            all_results.append({
                "example_id": example_id,
                "task_family": "3c",
                "dataset_name": "commonsense_qa",
                "dataset_item_id": dataset_item_id,
                "model_name": model_path,
                "prompting_method": method,
                "messages": messages,
                "expected_output": gold,
                "raw_model_output": raw_output,
                "response_time_sec": response_time_sec,
                "scores": {
                    "exact_match": exact,
                    "format_valid": format_valid
                },
                "annotation": {
                    "final_label": exact,
                    "notes": "" if exact == 1 else f"Parsed={parsed_pred}, expected={gold}"
                },
                "generation_config": GEN_CONFIG
            })

            example_id += 1

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    unload_model(model, tokenizer)

print(f"Saved to {OUTPUT_JSON}")

Loading model: Qwen2.5-3B


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: SmolLM3-3B


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Saved to csci5541-s26-hw5-ChengFu-3c.json


In [31]:
import pandas as pd

# =========================================
# Summary
# =========================================
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model_name": r["model_name"],
        "prompting_method": r["prompting_method"],
        "exact_match": r["scores"]["exact_match"],
        "format_valid": r["scores"]["format_valid"],
        "response_time_sec": r["response_time_sec"]
    })

summary_df = (
    pd.DataFrame(summary_rows)
      .groupby(["model_name", "prompting_method"])[["exact_match", "format_valid", "response_time_sec"]]
      .mean()
      .reset_index()
      .sort_values(["model_name", "prompting_method"])
)
summary_df["exact_match"] = summary_df["exact_match"].round(3)
summary_df["format_valid"] = summary_df["format_valid"].round(3)
summary_df["response_time_sec"] = summary_df["response_time_sec"].round(3)

print("\n=== Summary ===")
print(summary_df)
summary_df.to_csv("task3c_commonsenseqa_summary.csv", index=False)

print("Saved: task3c_commonsenseqa_summary.csv")


=== Summary ===
                 model_name prompting_method  exact_match  format_valid  \
0  HuggingFaceTB/SmolLM3-3B   plan-and-solve        0.000           0.0   
1  HuggingFaceTB/SmolLM3-3B      self-refine        0.667           1.0   
2  HuggingFaceTB/SmolLM3-3B        zero-shot        0.667           1.0   
3  Qwen/Qwen2.5-3B-Instruct   plan-and-solve        0.000           0.0   
4  Qwen/Qwen2.5-3B-Instruct      self-refine        0.700           1.0   
5  Qwen/Qwen2.5-3B-Instruct        zero-shot        0.700           1.0   

   response_time_sec  
0              0.131  
1              0.135  
2              0.129  
3              1.931  
4              0.131  
5              0.119  
Saved: task3c_commonsenseqa_summary.csv


In [38]:
import json
import random
import re
import gc
import time
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

TEAM_NAME = "ChengFu"
OUTPUT_JSON = f"csci5541-s26-hw5-{TEAM_NAME}-3c.json"
SEED = 42
NUM_EXAMPLES = 30

model_ids = {
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "SmolLM3-3B": "HuggingFaceTB/SmolLM3-3B"
}

prompting_methods = ["zero-shot", "plan-and-solve", "self-refine"]

GEN_CONFIG = {
    "temperature": 0.0,
    "top_p": 1.0,
    "max_new_tokens": 40,
    "num_generations": 1,
    "seed": SEED
}

random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset("tau/commonsense_qa")
selected_examples = dataset["validation"].select(range(NUM_EXAMPLES))


def build_zero_shot_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful assistant."},
        {"role": "user", "content": f"""Choose the best answer to the multiple-choice question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Output constraints:
- Answer with exactly one letter.
- Allowed outputs: A, B, C, D, or E.
- Do not explain.
- Output only the final answer.
"""}
    ]


def build_plan_and_solve_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful reasoning assistant."},
        {"role": "user", "content": f"""Solve the task in two stages.

Stage 1: Briefly compare the answer choices against the question.
Stage 2: Decide which option best answers the question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Output constraints:
- Exactly one letter
- Allowed outputs: A, B, C, D, or E
- No explanation
"""}
    ]


def build_self_refine_prompt(question, choices):
    return [
        {"role": "system", "content": "You are a careful assistant that checks its own output."},
        {"role": "user", "content": f"""Choose the best answer to the multiple-choice question.

Question:
{question}

Choices:
A. {choices["A"]}
B. {choices["B"]}
C. {choices["C"]}
D. {choices["D"]}
E. {choices["E"]}

Instructions:
1. Draft an answer internally.
2. Check whether the answer matches the question and choices.
3. Check whether the answer follows the output constraint.
4. Output only the final revised answer.

Output constraints:
- Exactly one letter
- Allowed outputs: A, B, C, D, or E
- No explanation
"""}
    ]


def build_messages(method, question, choices):
    if method == "zero-shot":
        return build_zero_shot_prompt(question, choices)
    if method == "plan-and-solve":
        return build_plan_and_solve_prompt(question, choices)
    if method == "self-refine":
        return build_self_refine_prompt(question, choices)
    raise ValueError(f"Unknown prompting method: {method}")


def load_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    return tokenizer, model


def generate_response(model, tokenizer, messages, max_new_tokens):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_time = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_length:]
    output_token_count = generated_tokens.shape[0]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    time_per_output_token_sec = (
        elapsed_time / output_token_count if output_token_count > 0 else None
    )

    return response, elapsed_time, output_token_count, time_per_output_token_sec


def strict_parse_choice(raw_output, method):
    text = raw_output.strip()
    if text in {"A", "B", "C", "D", "E"}:
        return text
    return None


def compute_scores(parsed_pred, gold):
    format_valid = int(parsed_pred in {"A", "B", "C", "D", "E"})
    exact_match = int(parsed_pred == gold) if parsed_pred is not None else 0
    return format_valid, exact_match


def unload_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


all_results = []
example_id = 1

for model_label, model_path in model_ids.items():
    print(f"Loading model: {model_label}")
    tokenizer, model = load_model_and_tokenizer(model_path)

    for i, ex in enumerate(selected_examples):
        question = ex["question"]
        choice_map = dict(zip(ex["choices"]["label"], ex["choices"]["text"]))
        choices = {
            "A": choice_map["A"],
            "B": choice_map["B"],
            "C": choice_map["C"],
            "D": choice_map["D"],
            "E": choice_map["E"],
        }
        gold = ex["answerKey"]
        dataset_item_id = f"commonsenseqa_val_{i:03d}"

        for method in prompting_methods:
            messages = build_messages(method, question, choices)

            try:
                raw_output, response_time_sec, output_token_count, time_per_output_token_sec = generate_response(
                    model=model,
                    tokenizer=tokenizer,
                    messages=messages,
                    max_new_tokens=GEN_CONFIG["max_new_tokens"]
                )
            except torch.cuda.OutOfMemoryError:
                print(f"OOM at model={model_label}, example={i}, method={method}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                raw_output = ""
                response_time_sec = None
                output_token_count = None
                time_per_output_token_sec = None

            parsed_pred = strict_parse_choice(raw_output, method)
            format_valid, exact = compute_scores(parsed_pred, gold)

            all_results.append({
                "example_id": example_id,
                "task_family": "3c",
                "dataset_name": "commonsense_qa",
                "dataset_item_id": dataset_item_id,
                "model_name": model_path,
                "prompting_method": method,
                "messages": messages,
                "expected_output": gold,
                "raw_model_output": raw_output,
                "response_time_sec": response_time_sec,
                "output_token_count": output_token_count,
                "time_per_output_token_sec": time_per_output_token_sec,
                "scores": {
                    "exact_match": exact,
                    "format_valid": format_valid
                },
                "annotation": {
                    "final_label": exact,
                    "notes": "" if exact == 1 else f"Parsed={parsed_pred}, expected={gold}"
                },
                "generation_config": GEN_CONFIG
            })

            example_id += 1

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    unload_model(model, tokenizer)

print(f"Saved to {OUTPUT_JSON}")


# =========================================
# Summary
# =========================================
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model_name": r["model_name"],
        "prompting_method": r["prompting_method"],
        "exact_match": r["scores"]["exact_match"],
        "format_valid": r["scores"]["format_valid"],
        "time_per_output_token_sec": r["time_per_output_token_sec"]
    })

summary_df = (
    pd.DataFrame(summary_rows)
      .groupby(["model_name", "prompting_method"])[
          ["exact_match", "format_valid", "time_per_output_token_sec"]
      ]
      .mean()
      .reset_index()
      .sort_values(["model_name", "prompting_method"])
)

summary_df["exact_match"] = summary_df["exact_match"].round(3)
summary_df["format_valid"] = summary_df["format_valid"].round(3)
summary_df["time_per_output_token_sec"] = summary_df["time_per_output_token_sec"].round(4)

print("\n=== Summary ===")
print(summary_df)

summary_df.to_csv("task3c_commonsenseqa_summary.csv", index=False)
print("Saved: task3c_commonsenseqa_summary.csv")

Loading model: Qwen2.5-3B


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: SmolLM3-3B


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Saved to csci5541-s26-hw5-ChengFu-3c.json

=== Summary ===
                 model_name prompting_method  exact_match  format_valid  \
0  HuggingFaceTB/SmolLM3-3B   plan-and-solve        0.633         1.000   
1  HuggingFaceTB/SmolLM3-3B      self-refine        0.667         1.000   
2  HuggingFaceTB/SmolLM3-3B        zero-shot        0.667         1.000   
3  Qwen/Qwen2.5-3B-Instruct   plan-and-solve        0.067         0.133   
4  Qwen/Qwen2.5-3B-Instruct      self-refine        0.700         1.000   
5  Qwen/Qwen2.5-3B-Instruct        zero-shot        0.700         1.000   

   time_per_output_token_sec  
0                     0.0650  
1                     0.0669  
2                     0.0641  
3                     0.0553  
4                     0.0649  
5                     0.0583  
Saved: task3c_commonsenseqa_summary.csv
